# 🎙️ Vietnamese-English Code-Switching ASR — Inference Demo

**Model:** Whisper large-v3 + LoRA fine-tuned for Vietnamese-English code-switching

**How to use:**
1. Upload your LoRA adapter folder (zip)
2. Upload your `.wav` audio file(s)
3. Run all cells

In [ ]:
# STEP 1: Install dependencies
!pip install -q transformers==4.40.0 peft==0.10.0 librosa

In [ ]:
# STEP 2: Imports
import os
import json
import torch
import librosa
import numpy as np
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
# STEP 3: Config — chỉnh paths ở đây

ADAPTER_ZIP   = "whisper-large-v3-lora-adapter.zip"  # upload file zip adapter
ADAPTER_DIR   = "whisper-large-v3-lora-adapter"      # tên folder sau khi giải nén
AUDIO_DIR     = "audio"                               # folder chứa file wav, hoặc dùng AUDIO_FILE bên dưới
AUDIO_FILE    = None                                  # ví dụ: "my_audio.wav" nếu chỉ test 1 file

BASE_MODEL_ID = "openai/whisper-large-v3"
SAMPLING_RATE = 16000
LANGUAGE      = "vi"
BATCH_SIZE    = 8

In [ ]:
# STEP 4: Unzip adapter
if os.path.exists(ADAPTER_ZIP):
    import zipfile
    with zipfile.ZipFile(ADAPTER_ZIP, 'r') as z:
        z.extractall(".")
    print(f"✅ Extracted {ADAPTER_ZIP}")
else:
    print(f"⚠️  {ADAPTER_ZIP} not found — skip if already extracted")

In [ ]:
# STEP 5: Load model
print("Loading base model...")
processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID)
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model = model.merge_and_unload()
model.eval()
print("✅ Model ready")

In [ ]:
# STEP 6: Helper functions
def load_audio(path, target_sr=SAMPLING_RATE):
    audio, _ = librosa.load(path, sr=target_sr, mono=True)
    max_samples = 30 * target_sr
    return audio[:max_samples] if len(audio) > max_samples else audio


def transcribe_batch(audio_paths, batch_size=BATCH_SIZE):
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task="transcribe")

    results = []
    for i in tqdm(range(0, len(audio_paths), batch_size), desc="Transcribing"):
        batch = audio_paths[i : i + batch_size]
        audios = [load_audio(p) for p in batch]

        inputs = processor.feature_extractor(
            audios, sampling_rate=SAMPLING_RATE,
            return_tensors="pt", padding="max_length", truncation=True,
        ).to(device)
        inputs["input_features"] = inputs["input_features"].to(dtype)

        with torch.no_grad():
            ids = model.generate(
                **inputs,
                forced_decoder_ids=forced_decoder_ids,
                max_new_tokens=225,
            )

        preds = processor.tokenizer.batch_decode(ids, skip_special_tokens=True)
        results.extend(preds)
    return results


print("Helper functions defined.")

In [ ]:
# STEP 7: Run inference
if AUDIO_FILE:
    audio_paths = [AUDIO_FILE]
else:
    audio_paths = sorted([
        os.path.join(AUDIO_DIR, f)
        for f in os.listdir(AUDIO_DIR)
        if f.endswith(".wav")
    ])

print(f"Found {len(audio_paths)} file(s)")

transcripts = transcribe_batch(audio_paths)

# Print results
results = {}
for path, text in zip(audio_paths, transcripts):
    filename = os.path.basename(path)
    results[filename] = {"text": text}
    print(f"{filename}: {text}")

In [ ]:
# STEP 8 (optional): Save results to JSON
OUTPUT_FILE = "transcripts.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅ Saved to {OUTPUT_FILE}")